# Mejores prácticas con Claude 4.X en producción

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/claude-4/04-mejores-practicas-claude4.ipynb)

Prompt Caching, retry logic, rate limiting y structured output para aplicaciones de producción.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
import time
import random
import json
from collections import defaultdict, deque

client = anthropic.Anthropic()

## 1. Prompt Caching

Cuando tienes un system prompt largo que se reutiliza, `cache_control` puede reducir el coste hasta un 90%.

In [ ]:
SYSTEM_LARGO = "Eres el asistente oficial de ACME Corp. " + ("Sigue estas normas de respuesta... " * 100)
print(f'System prompt: {len(SYSTEM_LARGO.split())} palabras')

def preguntar_con_cache(pregunta: str) -> dict:
    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=256,
        system=[
            {
                'type': 'text',
                'text': SYSTEM_LARGO,
                'cache_control': {'type': 'ephemeral'},
            }
        ],
        messages=[{'role': 'user', 'content': pregunta}],
    )
    return {
        'respuesta': response.content[0].text,
        'cache_read': getattr(response.usage, 'cache_read_input_tokens', 0),
        'cache_write': getattr(response.usage, 'cache_creation_input_tokens', 0),
        'input_tokens': response.usage.input_tokens,
    }

# Primera llamada: crea la caché
r1 = preguntar_con_cache('¿Cuál es vuestra política de devoluciones?')
print(f'1ª llamada — Cache write: {r1["cache_write"]}, Cache read: {r1["cache_read"]}')

# Segunda llamada: usa la caché
r2 = preguntar_con_cache('¿Tenéis envío gratuito?')
print(f'2ª llamada — Cache write: {r2["cache_write"]}, Cache read: {r2["cache_read"]}')
print(f'Tokens cacheados reutilizados: {r2["cache_read"]} (coste reducido ~90%)')

## 2. Retry logic con backoff exponencial

In [ ]:
def llamar_con_retry(prompt: str, modelo: str = 'claude-sonnet-4-6', max_intentos: int = 3) -> str:
    for intento in range(max_intentos):
        try:
            response = client.messages.create(
                model=modelo,
                max_tokens=512,
                messages=[{'role': 'user', 'content': prompt}],
            )
            return response.content[0].text
        except anthropic.RateLimitError:
            if intento == max_intentos - 1:
                raise
            espera = (2 ** intento) + random.uniform(0, 1)
            print(f'Rate limit. Esperando {espera:.1f}s (intento {intento+1}/{max_intentos})')
            time.sleep(espera)
        except anthropic.APIStatusError as e:
            if e.status_code == 529 and intento < max_intentos - 1:
                espera = (2 ** intento) * 5
                print(f'API sobrecargada. Esperando {espera}s')
                time.sleep(espera)
            else:
                raise
    raise RuntimeError('Todos los intentos fallaron')

respuesta = llamar_con_retry('¿Cuál es la capital de Japón?')
print(respuesta)

## 3. Structured output con tool use

In [ ]:
tool_clasificador = {
    'name': 'clasificar_email',
    'description': 'Clasifica y analiza un email entrante',
    'input_schema': {
        'type': 'object',
        'properties': {
            'urgente': {'type': 'boolean'},
            'categoria': {'type': 'string', 'enum': ['soporte', 'ventas', 'rrhh', 'spam', 'otro']},
            'resumen': {'type': 'string', 'maxLength': 100},
            'accion': {'type': 'string'},
        },
        'required': ['urgente', 'categoria', 'resumen', 'accion'],
    },
}

emails_test = [
    'URGENTE: El servidor de producción está caído desde hace 30 minutos. Clientes sin acceso.',
    'Nos gustaría conocer vuestros precios para una licencia corporativa de 50 usuarios.',
    '¡Has ganado un iPhone! Haz clic aquí para reclamarlo.',
]

for email in emails_test:
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        tools=[tool_clasificador],
        tool_choice={'type': 'tool', 'name': 'clasificar_email'},
        messages=[{'role': 'user', 'content': f'Clasifica: {email}'}],
    )
    result = next(b for b in response.content if b.type == 'tool_use')
    datos = result.input
    urgente = '🔴' if datos['urgente'] else '🟢'
    print(f'{urgente} [{datos["categoria"].upper()}] {datos["resumen"]}')

## Resumen de mejores prácticas

| Práctica | Impacto | Esfuerzo |
|----------|---------|----------|
| Prompt Caching | -90% coste en system prompts largos | Bajo |
| Retry con backoff | +99.9% disponibilidad | Bajo |
| Structured output | Parseo fiable 100% | Bajo |
| Router de modelos | -70% coste total | Medio |